# KPI 13 — Supplier Revenue + Concentration

**requirements.md (Supplier Performance, KPI 13):** "Supplier revenue + concentration (% from top N)."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_13_supplier_revenue_concentration` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cells below once (or after any change to the fact/dimension definitions).

**Definitions (state formulas):**

* **Grain:** one row per supplier in `dim_suppliers` (25 rows). Revenue is attributed per order line through `fact_order_items.supplier_sk`; all 25 suppliers sold on valid dates (verified 2026-11-20), so every row carries measures today — a future no-sales supplier keeps its row with NULL measures and ranks last (`NULL` = no sales, the fact convention).
* **Supplier identity:** `dim_suppliers` is thin by design — `supplier_name`, `country`, `rating` resolve via `company_id` → `dim_companies` **current SCD2 version** (`__END_AT IS NULL`). `rating` is nullable (1 supplier, Horizon Insumos, NULL); 0 suppliers share a `company_id`; 0 null names/countries.
* **Revenue bases = KPI 1:** gross = all statuses + `order_date_valid`; net = excluding `cancelled` + `order_date_valid`. `line_count` / `order_count` / `product_count` are valid-date gross; `units_sold` is net-basis `SUM(quantity)` (valid-date null-qty lines stay in revenue but not units — KPI 3/4 convention).
* **Concentration:** `net_revenue_rank` = ROW_NUMBER by net revenue DESC `NULLS LAST` (tie-break `supplier_id`); `net_revenue_share_pct` = ROUND(100·net / total net, 2) — the 25 shares sum to **99.99** (2-dp row-wise rounding artifact; KPI 3's 160 shares happen to round to 100.00); `cumulative_net_revenue_share_pct` = running share in rank order (100.00 at rank 25); `is_top_5_supplier` = rank ≤ 5 — the top-20% Pareto convention of KPI 2 (CEIL(0.2·25) = 5); `is_top_10_supplier` = rank ≤ 10.
* Consumers read "% from top N" straight off the view: **top 5 = 59.19%, top 10 = 80.11%** of net revenue; HHI ≈ 982 (headline cell).

**Verified 2026-11-20:** 25 rows = 25 distinct `supplier_id` = dim count; SUM(net) 28,333,498.48 / SUM(gross) 30,723,269.53 — EXACT KPI 1 tie; SUM(units_sold) 292,166 = KPI 4 tie; ranks contiguous 1..25; 5 top-5 / 10 top-10 flags; 0 null keys/names; 25/25 FULL match on all 12 per-supplier measures via independent recompute (correlated-subquery + self-join rank/cumulative path, FULL OUTER JOIN, 0 one-sided).

In [0]:
%sql
-- KPI 13 view: supplier revenue + concentration (% from top N). Idempotent.
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_13_supplier_revenue_concentration
TBLPROPERTIES (
  'comment' = 'KPI 13 - Supplier revenue + concentration, % from top N (requirements.md: Supplier Performance, KPI 13). Grain: one row per dim_suppliers supplier (25 rows); revenue attributed per order line via fact_order_items.supplier_sk; all 25 suppliers sold on valid dates (verified 2026-11-20), a future no-sales supplier keeps its row with NULL measures and ranks last (NULL = no sales, fact convention). Supplier identity: dim_suppliers is thin by design - supplier_name/country/rating resolve via company_id -> dim_companies CURRENT SCD2 version (__END_AT IS NULL); rating nullable (1 supplier, Horizon Insumos, NULL); 0 suppliers share a company_id; 0 null names/countries. Revenue bases = KPI 1: gross = all statuses + order_date_valid; net = excluding cancelled + order_date_valid. line_count/order_count/product_count are valid-date gross; units_sold is net-basis SUM(quantity) (valid-date null-qty lines stay in revenue but not units, KPI 3/4 convention). Concentration: net_revenue_rank = ROW_NUMBER by net revenue DESC NULLS LAST (tie-break supplier_id); net_revenue_share_pct = ROUND(100*net/total net,2) - the 25 shares sum to 99.99 (2-dp row-wise rounding artifact; KPI 3 shares happen to round to 100.00); cumulative_net_revenue_share_pct = running share in rank order (100.00 at last rank); is_top_5_supplier = rank <= 5 (top-20% Pareto, CEIL(0.2*25)=5, KPI 2 convention); is_top_10_supplier = rank <= 10. Verified 2026-11-20: 25 rows = 25 distinct supplier_id = dim count; SUM(net_revenue) 28,333,498.48 / SUM(gross_revenue) 30,723,269.53 EXACT KPI 1 tie; SUM(units_sold) 292,166 = KPI 4 tie; top 5 suppliers = 59.19% / top 10 = 80.11% of net revenue; 0 null supplier_sk/supplier_id/company_id/supplier_name/country; ranks contiguous 1..25; 5 top-5 / 10 top-10 flags; 25/25 FULL match on all 12 measures via independent recompute.'
)
AS
WITH valid_lines AS (
  SELECT supplier_sk, order_id, product_sk, quantity, status, line_total
  FROM b2b_ecommerce.gold.fact_order_items
  WHERE order_date_valid
),
supplier_agg AS (
  SELECT supplier_sk,
         COUNT(*) AS line_count,
         COUNT(DISTINCT order_id) AS order_count,
         COUNT(DISTINCT product_sk) AS product_count,
         SUM(CAST(CASE WHEN status <> 'cancelled' THEN quantity ELSE 0 END AS LONG)) AS units_sold,
         ROUND(SUM(CAST(line_total AS DOUBLE)), 2) AS gross_revenue,
         ROUND(SUM(CAST(CASE WHEN status <> 'cancelled' THEN line_total ELSE 0 END AS DOUBLE)), 2) AS net_revenue
  FROM valid_lines
  GROUP BY supplier_sk
),
supplier_base AS (
  SELECT s.supplier_sk, s.supplier_id, s.company_id,
         c.name AS supplier_name, c.country, c.rating,
         a.line_count, a.order_count, a.product_count, a.units_sold,
         a.gross_revenue, a.net_revenue
  FROM b2b_ecommerce.gold.dim_suppliers s
  LEFT JOIN b2b_ecommerce.gold.dim_companies c
    ON s.company_id = c.company_id AND c.__END_AT IS NULL
  LEFT JOIN supplier_agg a
    ON a.supplier_sk = s.supplier_sk
),
ranked AS (
  SELECT b.*,
         ROW_NUMBER() OVER (ORDER BY b.net_revenue DESC NULLS LAST, b.supplier_id ASC) AS net_revenue_rank,
         SUM(b.net_revenue) OVER () AS total_net_revenue,
         SUM(b.net_revenue) OVER (ORDER BY b.net_revenue DESC NULLS LAST, b.supplier_id ASC
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_net_revenue
  FROM supplier_base b
)
SELECT supplier_sk, supplier_id, company_id, supplier_name, country, rating,
       line_count, order_count, product_count, units_sold,
       gross_revenue, net_revenue,
       net_revenue_rank,
       ROUND(100 * net_revenue / total_net_revenue, 2) AS net_revenue_share_pct,
       ROUND(100 * cumulative_net_revenue / total_net_revenue, 2) AS cumulative_net_revenue_share_pct,
       net_revenue_rank <= 5 AS is_top_5_supplier,
       net_revenue_rank <= 10 AS is_top_10_supplier
FROM ranked

In [0]:
%sql
-- Verification: shape, totals, integrity. Expected: 25 rows = 25 distinct
-- supplier_id = dim_suppliers count (all suppliers sell on valid dates);
-- SUM(net_revenue) 28,333,498.48 and SUM(gross_revenue) 30,723,269.53 =
-- EXACT KPI 1 tie; SUM(units_sold) 292,166 = KPI 4 units tie; shares sum to
-- 99.99 (2-dp row-wise rounding artifact); cumulative share = 100.00 at
-- rank 25; ranks contiguous 1..25; 5 top-5 / 10 top-10 flags with top-5 net
-- 16,771,297.18 = 59.19% and top-10 = 80.11% of net; 0 null keys, names,
-- countries or measures.
SELECT
  COUNT(*) AS rows_suppliers,
  COUNT(DISTINCT supplier_id) AS distinct_supplier_ids,
  (SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_suppliers) AS dim_supplier_count,
  ROUND(SUM(net_revenue), 2) AS total_net,
  ROUND(SUM(gross_revenue), 2) AS total_gross,
  (SELECT ROUND(SUM(net_revenue), 2) FROM b2b_ecommerce.gold.kpi_1_monthly_revenue_trend) AS kpi1_net_check,
  (SELECT ROUND(SUM(gross_revenue), 2) FROM b2b_ecommerce.gold.kpi_1_monthly_revenue_trend) AS kpi1_gross_check,
  SUM(units_sold) AS total_units,
  (SELECT SUM(units_sold) FROM b2b_ecommerce.gold.kpi_4_gross_margin) AS kpi4_units_check,
  ROUND(SUM(net_revenue_share_pct), 2) AS sum_shares_pct,
  MAX(CASE WHEN net_revenue_rank = 25 THEN cumulative_net_revenue_share_pct END) AS cum_share_at_rank25,
  MIN(net_revenue_rank) AS min_rank,
  MAX(net_revenue_rank) AS max_rank,
  COUNT(DISTINCT net_revenue_rank) AS distinct_ranks,
  SUM(CAST(is_top_5_supplier AS INT)) AS top5_flag_count,
  SUM(CAST(is_top_10_supplier AS INT)) AS top10_flag_count,
  ROUND(SUM(CASE WHEN is_top_5_supplier THEN net_revenue END), 2) AS top5_net,
  ROUND(100 * SUM(CASE WHEN is_top_5_supplier THEN net_revenue END) / SUM(net_revenue), 2) AS top5_share_pct,
  ROUND(100 * SUM(CASE WHEN is_top_10_supplier THEN net_revenue END) / SUM(net_revenue), 2) AS top10_share_pct,
  SUM(CAST(supplier_sk IS NULL OR supplier_id IS NULL OR company_id IS NULL AS INT)) AS null_keys,
  SUM(CAST(supplier_name IS NULL OR country IS NULL AS INT)) AS null_names_countries,
  SUM(CAST(line_count IS NULL OR net_revenue IS NULL OR units_sold IS NULL AS INT)) AS null_measures
FROM b2b_ecommerce.gold.kpi_13_supplier_revenue_concentration

In [0]:
%sql
-- Verification: independently recompute every supplier row through a
-- different path - measures via per-supplier correlated scalar subqueries
-- on fact_order_items (vs the view's pre-aggregated CTE + LEFT JOIN), rank
-- and cumulative revenue via self-joins on the sort key (net DESC,
-- supplier_id ASC - vs the view's window functions) - then FULL OUTER JOIN
-- against the stored view and compare all 12 per-supplier measures.
-- Expected: 25 rows compared, 0 only on either side, 0 mismatches on every
-- measure including rank, share, cumulative share and both top-N flags.
-- NULL-vs-0 note: a correlated SUM returns NULL for a no-sales supplier
-- (the view's convention) while COUNT would return 0 - not exercised today
-- (all 25 sell) but the <=> comparison would need mapping if it ever is.
WITH recompute AS (
  SELECT s.supplier_id,
         (SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_order_items f
           WHERE f.supplier_sk = s.supplier_sk AND f.order_date_valid) AS line_count,
         (SELECT COUNT(DISTINCT f.order_id) FROM b2b_ecommerce.gold.fact_order_items f
           WHERE f.supplier_sk = s.supplier_sk AND f.order_date_valid) AS order_count,
         (SELECT COUNT(DISTINCT f.product_sk) FROM b2b_ecommerce.gold.fact_order_items f
           WHERE f.supplier_sk = s.supplier_sk AND f.order_date_valid) AS product_count,
         (SELECT SUM(CAST(CASE WHEN f.status <> 'cancelled' THEN f.quantity ELSE 0 END AS LONG))
            FROM b2b_ecommerce.gold.fact_order_items f
           WHERE f.supplier_sk = s.supplier_sk AND f.order_date_valid) AS units_sold,
         (SELECT ROUND(SUM(CAST(f.line_total AS DOUBLE)), 2)
            FROM b2b_ecommerce.gold.fact_order_items f
           WHERE f.supplier_sk = s.supplier_sk AND f.order_date_valid) AS gross_revenue,
         (SELECT ROUND(SUM(CAST(CASE WHEN f.status <> 'cancelled' THEN f.line_total ELSE 0 END AS DOUBLE)), 2)
            FROM b2b_ecommerce.gold.fact_order_items f
           WHERE f.supplier_sk = s.supplier_sk AND f.order_date_valid) AS net_revenue
  FROM b2b_ecommerce.gold.dim_suppliers s
),
ranked AS (
  SELECT r.*,
         (SELECT COUNT(*) + 1 FROM recompute r2
           WHERE r2.net_revenue > r.net_revenue
              OR (r2.net_revenue = r.net_revenue AND r2.supplier_id < r.supplier_id)) AS rank_recompute,
         (SELECT ROUND(SUM(r2.net_revenue), 2) FROM recompute r2) AS total_net,
         (SELECT ROUND(SUM(r2.net_revenue), 2) FROM recompute r2
           WHERE r2.net_revenue > r.net_revenue
              OR (r2.net_revenue = r.net_revenue AND r2.supplier_id <= r.supplier_id)) AS cumulative_net
  FROM recompute r
),
compare AS (
  SELECT v.supplier_id AS v_id, v.line_count AS v_line_count, v.order_count AS v_order_count,
         v.product_count AS v_product_count, v.units_sold AS v_units, v.gross_revenue AS v_gross,
         v.net_revenue AS v_net, v.net_revenue_rank AS v_rank,
         v.net_revenue_share_pct AS v_share, v.cumulative_net_revenue_share_pct AS v_cum_share,
         v.is_top_5_supplier AS v_top5, v.is_top_10_supplier AS v_top10,
         rc.supplier_id AS r_id, rc.line_count AS r_line_count, rc.order_count AS r_order_count,
         rc.product_count AS r_product_count, rc.units_sold AS r_units, rc.gross_revenue AS r_gross,
         rc.net_revenue AS r_net, rc.rank_recompute AS r_rank,
         ROUND(100 * rc.net_revenue / rc.total_net, 2) AS r_share,
         ROUND(100 * rc.cumulative_net / rc.total_net, 2) AS r_cum_share,
         rc.rank_recompute <= 5 AS r_top5, rc.rank_recompute <= 10 AS r_top10
  FROM b2b_ecommerce.gold.kpi_13_supplier_revenue_concentration v
  FULL OUTER JOIN ranked rc ON v.supplier_id = rc.supplier_id
)
SELECT
  COUNT(*) AS compared_rows,
  SUM(CAST(v_id IS NULL AS INT)) AS only_in_view,
  SUM(CAST(r_id IS NULL AS INT)) AS only_in_recompute,
  SUM(CAST(NOT (v_line_count <=> r_line_count) AS INT)) AS line_count_mismatch,
  SUM(CAST(NOT (v_order_count <=> r_order_count) AS INT)) AS order_count_mismatch,
  SUM(CAST(NOT (v_product_count <=> r_product_count) AS INT)) AS product_count_mismatch,
  SUM(CAST(NOT (v_units <=> r_units) AS INT)) AS units_mismatch,
  SUM(CAST(NOT (v_gross <=> r_gross) AS INT)) AS gross_mismatch,
  SUM(CAST(NOT (v_net <=> r_net) AS INT)) AS net_mismatch,
  SUM(CAST(NOT (v_rank <=> r_rank) AS INT)) AS rank_mismatch,
  SUM(CAST(NOT (v_share <=> r_share) AS INT)) AS share_mismatch,
  SUM(CAST(NOT (v_cum_share <=> r_cum_share) AS INT)) AS cumulative_share_mismatch,
  SUM(CAST(NOT (v_top5 <=> r_top5) AS INT)) AS top5_flag_mismatch,
  SUM(CAST(NOT (v_top10 <=> r_top10) AS INT)) AS top10_flag_mismatch
FROM compare

In [0]:
%sql
-- Headline: top-10 suppliers by net revenue with share + cumulative share,
-- plus the concentration constants (window aggregates on every row):
-- supplier_count 25, total net 28,333,498.48, top-5 net share 59.19%,
-- top-10 80.11%, HHI ~982 (unconcentrated by the 1500/2500 thresholds).
SELECT net_revenue_rank, supplier_id, supplier_name, country, rating,
       order_count, product_count, units_sold, net_revenue,
       net_revenue_share_pct, cumulative_net_revenue_share_pct,
       is_top_5_supplier,
       COUNT(*) OVER () AS supplier_count,
       ROUND(SUM(net_revenue) OVER (), 2) AS total_net_revenue,
       ROUND(100 * SUM(CASE WHEN is_top_5_supplier THEN net_revenue ELSE 0 END) OVER ()
             / SUM(net_revenue) OVER (), 2) AS top5_net_share_pct,
       ROUND(100 * SUM(CASE WHEN is_top_10_supplier THEN net_revenue ELSE 0 END) OVER ()
             / SUM(net_revenue) OVER (), 2) AS top10_net_share_pct,
       ROUND(SUM(net_revenue * net_revenue) OVER ()
             / (SUM(net_revenue) OVER () * SUM(net_revenue) OVER ()) * 10000, 1) AS hhi_index
FROM b2b_ecommerce.gold.kpi_13_supplier_revenue_concentration
ORDER BY net_revenue_rank
LIMIT 10